# 📋 KoboToolbox MEL Data Pipeline
### Health Programme Monitoring — Data Collection to Analysis

**Author:** Pantouin Adjinsala  
**Affiliation:** University Lecturer & Civic Tech Contributor, AfricTivistes CitizenLab Cameroon  
**Context:** Monitoring, Evaluation and Learning (MEL) teams across Africa rely on KoboToolbox
for field data collection. This notebook demonstrates a full pipeline: pulling data from the
KoboToolbox API, cleaning and validating it, computing programme indicators, and producing
MEL-ready outputs for reporting — using a simulated maternal and child health programme as
the working example.

---

## 📌 Contents
1. Setup & Imports
2. KoboToolbox API Reference (live connection)
3. Simulated Dataset Generation
4. Data Cleaning & Validation
5. Descriptive Analysis & Disaggregation
6. MEL Indicator Computation
7. Visualisations
8. Export to Excel
9. Discussion & Adaptations

## 1. Setup & Imports

In [ ]:
!pip install openpyxl xlsxwriter requests -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import requests, json, warnings
from datetime import datetime, timedelta
from io import BytesIO

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded successfully.')

## 2. KoboToolbox API Reference

This section is a **plug-and-play reference** for teams with live KoboToolbox projects.
Replace the placeholders with your credentials and form UID to pull real data.
The rest of the pipeline works identically whether data comes from the API or simulation.

> **Skip to Section 3** if you want to run the notebook immediately with simulated data.

In [ ]:
# ─────────────────────────────────────────────────────
# KOBO API CONFIGURATION  (replace with your values)
# ─────────────────────────────────────────────────────
KOBO_TOKEN  = 'YOUR_API_TOKEN_HERE'      # Account > Security > API token
FORM_UID    = 'YOUR_FORM_UID_HERE'       # Projects > form > Settings > Sharing
KOBO_SERVER = 'https://kf.kobotoolbox.org'  # or https://kobo.humanitarianresponse.info

def fetch_kobo_data(token, form_uid, server=KOBO_SERVER):
    """
    Fetch all submissions from a KoboToolbox form.
    Returns a cleaned pandas DataFrame.
    """
    url     = f'{server}/api/v2/assets/{form_uid}/data/?format=json'
    headers = {'Authorization': f'Token {token}'}

    all_results, next_url = [], url
    while next_url:
        resp = requests.get(next_url, headers=headers, timeout=30)
        resp.raise_for_status()
        payload    = resp.json()
        all_results.extend(payload.get('results', []))
        next_url   = payload.get('next')   # pagination

    df = pd.json_normalize(all_results)

    # Strip KoboToolbox internal prefix from column names (e.g. 'group_abc/q1' -> 'q1')
    df.columns = [c.split('/')[-1] for c in df.columns]

    print(f'Fetched {len(df):,} submissions | {df.shape[1]} fields')
    return df

# Uncomment to run with real credentials:
# df_live = fetch_kobo_data(KOBO_TOKEN, FORM_UID)
# df_live.head()
print('API reference ready. Running with simulated data below.')

## 3. Simulated Dataset Generation

We generate 500 realistic survey submissions for a **Maternal & Child Health (MCH)**
programme across Cameroon's 10 regions. Fields mirror a typical KoboToolbox form output.

In [ ]:
np.random.seed(42)
N = 500

REGIONS = {
    'Centre':     {'weight': 0.18, 'urban_rate': 0.65},
    'Littoral':   {'weight': 0.15, 'urban_rate': 0.72},
    'West':       {'weight': 0.13, 'urban_rate': 0.45},
    'North West': {'weight': 0.10, 'urban_rate': 0.38},
    'South West': {'weight': 0.09, 'urban_rate': 0.42},
    'Adamawa':    {'weight': 0.08, 'urban_rate': 0.33},
    'North':      {'weight': 0.09, 'urban_rate': 0.28},
    'Far North':  {'weight': 0.10, 'urban_rate': 0.22},
    'East':       {'weight': 0.05, 'urban_rate': 0.30},
    'South':      {'weight': 0.03, 'urban_rate': 0.35},
}

region_names   = list(REGIONS.keys())
region_weights = [v['weight'] for v in REGIONS.values()]
urban_rates    = {k: v['urban_rate'] for k, v in REGIONS.items()}

regions  = np.random.choice(region_names, size=N, p=region_weights)
is_urban = np.array([np.random.rand() < urban_rates[r] for r in regions])

# Survey dates — spread over 12 months
base_date   = datetime(2023, 1, 1)
survey_days = np.random.randint(0, 365, N)
survey_dates = [base_date + timedelta(days=int(d)) for d in survey_days]

# Respondent demographics
age_groups = np.random.choice(
    ['15-19','20-24','25-29','30-34','35-49'],
    size=N, p=[0.10, 0.28, 0.30, 0.20, 0.12]
)
education = np.random.choice(
    ['No formal','Primary','Secondary','Tertiary'],
    size=N, p=[0.18, 0.32, 0.38, 0.12]
)

# Health indicators (urban areas score better)
anc_base = np.where(is_urban, 0.82, 0.58)
anc4_plus = np.random.binomial(1, anc_base)

skilled_base = np.where(is_urban, 0.88, 0.55)
skilled_delivery = np.random.binomial(1, skilled_base)

vacc_base = np.where(is_urban, 0.78, 0.52)
full_vaccination = np.random.binomial(1, vacc_base)

postnatal_base = np.where(is_urban, 0.70, 0.44)
postnatal_check = np.random.binomial(1, postnatal_base)

# Health facility distance (km)
distance_km = np.where(
    is_urban,
    np.random.exponential(2, N),
    np.random.exponential(12, N)
).round(1)

# Health worker interaction
hw_visit = np.random.binomial(1, np.where(is_urban, 0.60, 0.35))

# Satisfaction (1-5 Likert)
satisfaction = np.random.choice([1,2,3,4,5], N, p=[0.05,0.10,0.20,0.40,0.25])

# Data quality issues (intentional — for cleaning demo)
submission_ids = [f'SUB-{1000+i}' for i in range(N)]
# Inject 15 duplicate IDs
dup_idx = np.random.choice(N, 15, replace=False)
for i in dup_idx:
    submission_ids[i] = submission_ids[dup_idx[0]]

# Inject 20 out-of-range ages
age_raw = np.random.randint(15, 50, N).astype(float)
outlier_idx = np.random.choice(N, 20, replace=False)
age_raw[outlier_idx] = np.random.choice([5, 99, -1], 20)

df = pd.DataFrame({
    '_id'              : submission_ids,
    'submission_date'  : survey_dates,
    'region'           : regions,
    'setting'          : np.where(is_urban, 'Urban', 'Rural'),
    'age'              : age_raw,
    'age_group'        : age_groups,
    'education_level'  : education,
    'anc4_plus'        : anc4_plus,
    'skilled_delivery' : skilled_delivery,
    'full_vaccination' : full_vaccination,
    'postnatal_check'  : postnatal_check,
    'hw_visit_last_3m' : hw_visit,
    'distance_facility_km': distance_km,
    'satisfaction_score': satisfaction,
})

print(f'Simulated dataset: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

## 4. Data Cleaning & Validation Pipeline

A production MEL pipeline must handle duplicates, out-of-range values, and
missing data systematically before any analysis. We log every issue found.

In [ ]:
class MELCleaner:
    """Reproducible cleaning pipeline for KoboToolbox MEL survey data."""

    def __init__(self, df):
        self.df  = df.copy()
        self.log = []

    def check_duplicates(self, id_col='_id'):
        dupes = self.df.duplicated(subset=[id_col], keep='first')
        n = dupes.sum()
        self.df = self.df[~dupes].reset_index(drop=True)
        self.log.append(f'Duplicates removed    : {n}')
        return self

    def validate_range(self, col, min_val, max_val, fill=np.nan):
        mask  = ~self.df[col].between(min_val, max_val)
        n     = mask.sum()
        self.df.loc[mask, col] = fill
        self.log.append(f'Out-of-range [{col}]: {n} set to {fill}')
        return self

    def fill_missing(self, col, strategy='mode'):
        n = self.df[col].isna().sum()
        if n == 0:
            return self
        if strategy == 'mode':
            fill_val = self.df[col].mode()[0]
        elif strategy == 'median':
            fill_val = self.df[col].median()
        self.df[col] = self.df[col].fillna(fill_val)
        self.log.append(f'Missing filled [{col}]: {n} using {strategy} ({fill_val})')
        return self

    def parse_dates(self, col):
        self.df[col] = pd.to_datetime(self.df[col], errors='coerce')
        self.df['month']   = self.df[col].dt.to_period('M').astype(str)
        self.df['quarter'] = self.df[col].dt.to_period('Q').astype(str)
        self.log.append(f'Date parsed [{col}]: month & quarter columns added')
        return self

    def report(self):
        print('=== CLEANING REPORT ===')
        for entry in self.log:
            print(' >', entry)
        print(f'Final dataset        : {self.df.shape[0]} rows x {self.df.shape[1]} columns')
        return self.df


cleaner = (
    MELCleaner(df)
    .check_duplicates(id_col='_id')
    .validate_range('age', min_val=15, max_val=49)
    .fill_missing('age', strategy='median')
    .parse_dates('submission_date')
)
df_clean = cleaner.report()

## 5. Descriptive Analysis & Disaggregation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Submissions by region
rc = df_clean['region'].value_counts().sort_values()
axes[0,0].barh(rc.index, rc.values, color=sns.color_palette('Blues_d', len(rc)))
axes[0,0].set_title('Submissions by Region', fontweight='bold')
axes[0,0].set_xlabel('Count')

# Urban vs Rural split
sc = df_clean['setting'].value_counts()
axes[0,1].pie(sc.values, labels=sc.index, autopct='%1.1f%%',
              colors=['#3498db','#e67e22'], startangle=90,
              wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[0,1].set_title('Urban vs Rural Split', fontweight='bold')

# Age group distribution
order = ['15-19','20-24','25-29','30-34','35-49']
ag = df_clean['age_group'].value_counts().reindex(order)
axes[1,0].bar(ag.index, ag.values, color=sns.color_palette('Greens_d', len(ag)))
axes[1,0].set_title('Age Group Distribution', fontweight='bold')
axes[1,0].set_xlabel('Age Group')
axes[1,0].set_ylabel('Count')

# Satisfaction distribution
sv = df_clean['satisfaction_score'].value_counts().sort_index()
bars = axes[1,1].bar(sv.index, sv.values,
                     color=['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60'])
axes[1,1].set_title('Service Satisfaction (1-5 Likert)', fontweight='bold')
axes[1,1].set_xlabel('Score')
axes[1,1].set_ylabel('Count')
axes[1,1].set_xticks([1,2,3,4,5])
axes[1,1].set_xticklabels(['1\nVery Poor','2','3\nNeutral','4','5\nExcellent'])

plt.suptitle('Survey Overview — Maternal & Child Health Programme', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('survey_overview.png', bbox_inches='tight')
plt.show()

## 6. MEL Indicator Computation

We compute the four core programme indicators against set targets,
then disaggregate by region and urban/rural setting.

In [ ]:
TARGETS = {
    'ANC 4+ Coverage'        : 0.75,
    'Skilled Delivery Rate'  : 0.80,
    'Full Vaccination Rate'  : 0.70,
    'Postnatal Check Rate'   : 0.65,
}

INDICATOR_COLS = {
    'ANC 4+ Coverage'       : 'anc4_plus',
    'Skilled Delivery Rate' : 'skilled_delivery',
    'Full Vaccination Rate' : 'full_vaccination',
    'Postnatal Check Rate'  : 'postnatal_check',
}

# Overall indicator table
records = []
for name, col in INDICATOR_COLS.items():
    actual = df_clean[col].mean()
    target = TARGETS[name]
    records.append({
        'Indicator'  : name,
        'Actual (%)'  : round(actual * 100, 1),
        'Target (%)'  : round(target * 100, 1),
        'Gap (pp)'    : round((actual - target) * 100, 1),
        'Status'      : 'On track' if actual >= target else 'Below target'
    })

indicators_df = pd.DataFrame(records)
print('=== PROGRAMME INDICATOR SUMMARY ===')
print(indicators_df.to_string(index=False))

In [ ]:
# Disaggregation by region x indicator
disagg = df_clean.groupby('region')[list(INDICATOR_COLS.values())].mean() * 100
disagg.columns = list(INDICATOR_COLS.keys())
disagg = disagg.round(1).reset_index().sort_values('ANC 4+ Coverage')
print('\nRegional Disaggregation:')
disagg

## 7. Visualisations

In [ ]:
# Target vs Actual bullet chart
fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(indicators_df))
bars  = ax.barh(y_pos, indicators_df['Actual (%)'],
                color=['#2ecc71' if s == 'On track' else '#e74c3c'
                       for s in indicators_df['Status']],
                height=0.5, label='Actual')
for i, row in indicators_df.iterrows():
    ax.plot([row['Target (%)'], row['Target (%)']],
            [i - 0.35, i + 0.35], color='#2c3e50', lw=2.5, label='Target' if i == 0 else '')
    ax.text(row['Actual (%)'] + 0.5, i, f"{row['Actual (%)']}%", va='center', fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(indicators_df['Indicator'])
ax.set_xlabel('Coverage (%)')
ax.set_title('Programme Indicators — Actual vs Target', fontweight='bold')
ax.set_xlim(0, 105)
ax.legend(loc='lower right')
ax.axvline(x=0, color='grey', lw=0.5)
plt.tight_layout()
plt.savefig('indicators_vs_target.png', bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: regional disaggregation
fig, ax = plt.subplots(figsize=(11, 6))
heat_data = disagg.set_index('region')[list(INDICATOR_COLS.keys())]
sns.heatmap(
    heat_data, annot=True, fmt='.1f', cmap='RdYlGn',
    linewidths=0.5, ax=ax, vmin=30, vmax=100,
    cbar_kws={'label': 'Coverage (%)'}
)
ax.set_title('Regional Disaggregation — All Indicators (%)', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('regional_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Urban vs Rural comparison
ur = df_clean.groupby('setting')[list(INDICATOR_COLS.values())].mean() * 100
ur.columns = list(INDICATOR_COLS.keys())
ur = ur.T.reset_index()
ur.columns = ['Indicator', 'Rural', 'Urban']

x    = np.arange(len(ur))
w    = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, ur['Urban'], w, label='Urban', color='#3498db')
ax.bar(x + w/2, ur['Rural'], w, label='Rural', color='#e67e22')
for i, row in ur.iterrows():
    ax.plot([i - w/2 - 0.05, i + w/2 + 0.05],
            [TARGETS[row['Indicator']]*100, TARGETS[row['Indicator']]*100],
            'k--', lw=1.2, alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(ur['Indicator'], rotation=15, ha='right')
ax.set_ylabel('Coverage (%)')
ax.set_title('Urban vs Rural Coverage — All Indicators (dashed = target)', fontweight='bold')
ax.legend()
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('urban_rural_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# Monthly trend for ANC 4+
monthly = (df_clean.groupby('month')['anc4_plus']
           .agg(['mean','count'])
           .reset_index())
monthly.columns = ['month','anc4_rate','submissions']
monthly['anc4_rate'] *= 100

fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()
ax1.plot(monthly['month'], monthly['anc4_rate'],
         color='#2c3e50', lw=2, marker='o', markersize=5, label='ANC 4+ Rate')
ax1.axhline(y=75, color='red', ls='--', lw=1.2, label='Target (75%)')
ax2.bar(monthly['month'], monthly['submissions'],
        color='#bdc3c7', alpha=0.5, label='Submissions')
ax1.set_ylabel('ANC 4+ Coverage (%)', color='#2c3e50')
ax2.set_ylabel('Submissions', color='grey')
ax1.set_title('Monthly ANC 4+ Coverage vs Submissions', fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.set_ylim(0, 105)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')
plt.tight_layout()
plt.savefig('monthly_trend.png', bbox_inches='tight')
plt.show()

## 8. Export to Excel (MEL-Ready Output)

We export a multi-sheet Excel workbook ready for sharing with programme teams,
donors, or government partners — the standard deliverable in MEL workflows.

In [ ]:
output_file = 'mch_mel_report.xlsx'

with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    wb_xl = writer.book

    # Formats
    header_fmt = wb_xl.add_format({
        'bold': True, 'bg_color': '#2c3e50', 'font_color': 'white',
        'border': 1, 'align': 'center'
    })
    green_fmt = wb_xl.add_format({'bg_color': '#d5f5e3', 'border': 1})
    red_fmt   = wb_xl.add_format({'bg_color': '#fadbd8', 'border': 1})
    pct_fmt   = wb_xl.add_format({'num_format': '0.0"%"', 'border': 1})

    # Sheet 1 — Indicator Summary
    indicators_df.to_excel(writer, sheet_name='Indicator Summary', index=False)
    ws = writer.sheets['Indicator Summary']
    for col_num, col_name in enumerate(indicators_df.columns):
        ws.write(0, col_num, col_name, header_fmt)
    ws.set_column('A:A', 28)
    ws.set_column('B:E', 16)

    # Sheet 2 — Regional Disaggregation
    disagg.to_excel(writer, sheet_name='Regional Disaggregation', index=False)
    ws2 = writer.sheets['Regional Disaggregation']
    for col_num, col_name in enumerate(disagg.columns):
        ws2.write(0, col_num, col_name, header_fmt)
    ws2.set_column('A:A', 18)
    ws2.set_column('B:E', 22)

    # Sheet 3 — Urban vs Rural
    ur.to_excel(writer, sheet_name='Urban vs Rural', index=False)
    ws3 = writer.sheets['Urban vs Rural']
    for col_num, col_name in enumerate(ur.columns):
        ws3.write(0, col_num, col_name, header_fmt)
    ws3.set_column('A:C', 24)

    # Sheet 4 — Raw Cleaned Data
    df_clean.to_excel(writer, sheet_name='Cleaned Data', index=False)
    ws4 = writer.sheets['Cleaned Data']
    for col_num, col_name in enumerate(df_clean.columns):
        ws4.write(0, col_num, col_name, header_fmt)
    ws4.set_column('A:P', 20)

print(f'Excel report exported: {output_file}')
print('Sheets: Indicator Summary | Regional Disaggregation | Urban vs Rural | Cleaned Data')

## 9. Discussion & Adaptations

### What this pipeline demonstrates
A complete, reproducible MEL data workflow: from raw KoboToolbox submission
to cleaned data, indicator computation, disaggregated analysis, and formatted
Excel output — all in a single auditable notebook.

### Adapting to your own KoboToolbox project
1. Replace credentials in **Section 2** and call `fetch_kobo_data()`
2. Update `INDICATOR_COLS` and `TARGETS` in **Section 6** to match your logframe
3. Update `REGIONS` in **Section 3** if using a different country context
4. The `MELCleaner` class in **Section 4** is reusable across projects — extend it with domain-specific rules as needed

### Planned Extensions
1. **Automated PDF report** — generate a formatted PDF from charts and tables using `reportlab` or `weasyprint`
2. **Outlier detection** — flag enumerator-level anomalies (suspiciously fast submissions, GPS drift)
3. **Streamlit dashboard** — deploy as a live, shareable web dashboard for programme teams
4. **Multi-round tracking** — extend to compare baseline, midline, and endline submissions

---

> *This pipeline was developed based on real MEL experience with KoboToolbox across civic and health programmes in Cameroon.*  
> *Adapt freely for your own monitoring and evaluation workflows.*

**References**
- KoboToolbox API documentation: https://support.kobotoolbox.org/api.html
- WHO health indicator definitions: https://www.who.int/data/gho/indicator-metadata-registry
- USAID MEL standards: https://usaidlearninglab.org